# 유방암 이진분류 실습
**AIsquare Lab** | PyTorch 버전

▶ 아래 셀의 실행 버튼을 누르세요!

In [ ]:
# -*- coding: utf-8 -*-
# 유방암 이진분류 실습 — PyTorch 버전 (AIsquare Lab · AI의 정석)
# 이 코드를 Colab에 붙여넣고 ▶ 실행하세요!

import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print(f"PyTorch {torch.__version__} (CPU 모드)")

# ============================================================
# 1. 데이터 준비
# ============================================================
bc = load_breast_cancer()
X = bc.data[:, [20, 29]]  # 세포 크기, 세포 복잡도
y = bc.target              # 1=양성, 0=악성
print(f"전체 데이터: {len(y)}명 (양성 {(y==1).sum()}, 악성 {(y==0).sum()})")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print(f"Train: {len(y_train)}명, Test: {len(y_test)}명")

# numpy → torch tensor
X_tr = torch.tensor(X_train_s, dtype=torch.float32)
y_tr = torch.tensor(y_train, dtype=torch.float32)
X_te = torch.tensor(X_test_s, dtype=torch.float32)
y_te = torch.tensor(y_test, dtype=torch.float32)

# ============================================================
# 2. 모델: 로지스틱 회귀 (Linear + Sigmoid)
# ============================================================
model = torch.nn.Linear(2, 1)          # w(2개) + b(1개)
torch.nn.init.zeros_(model.weight)
torch.nn.init.zeros_(model.bias)

criterion = torch.nn.BCEWithLogitsLoss()  # sigmoid + BCE 통합
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# ============================================================
# 3. 학습: Batch 경사하강법 500 에폭
# ============================================================
loss_h, acc_h, tacc_h = [], [], []

print("\n학습 시작!")
for epoch in range(500):
    # Forward
    logits = model(X_tr).squeeze()
    loss = criterion(logits, y_tr)

    # Backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # 기록
    with torch.no_grad():
        loss_h.append(loss.item())
        preds = (torch.sigmoid(logits) >= 0.5).float()
        acc_h.append((preds == y_tr).float().mean().item() * 100)

        t_logits = model(X_te).squeeze()
        t_preds = (torch.sigmoid(t_logits) >= 0.5).float()
        tacc_h.append((t_preds == y_te).float().mean().item() * 100)

    if epoch % 100 == 0:
        print(f"  Epoch {epoch:3d}: Loss={loss_h[-1]:.4f}, "
              f"Train={acc_h[-1]:.1f}%, Test={tacc_h[-1]:.1f}%")

print(f"\n✅ 최종 결과: Train {acc_h[-1]:.1f}%, Test {tacc_h[-1]:.1f}%")

# 학습된 파라미터
w = model.weight.data.squeeze().numpy()
b_val = model.bias.data.item()
print(f"   w = [{w[0]:.4f}, {w[1]:.4f}], b = {b_val:.4f}")

# ============================================================
# 4. 시각화
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Loss
axes[0].plot(loss_h, color='gold', lw=2)
axes[0].set_title('Loss (BCE)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')

# Accuracy
axes[1].plot(acc_h, color='green', lw=2, label='Train')
axes[1].plot(tacc_h, color='orange', lw=2, ls='--', label='Test')
axes[1].set_title('Accuracy (%)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].legend()

# Decision Boundary
benign, malign = y == 1, y == 0
axes[2].scatter(X[benign, 0], X[benign, 1], c='green', s=25, alpha=0.5,
                label=f'Benign ({benign.sum()})')
axes[2].scatter(X[malign, 0], X[malign, 1], c='red', s=25, alpha=0.5,
                label=f'Malignant ({malign.sum()})')
mean, std = scaler.mean_, scaler.scale_
xr = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200)
xb = -(w[0] / std[0]) / (w[1] / std[1]) * xr + \
     (w[0] * mean[0] / std[0] + w[1] * mean[1] / std[1] - b_val) / (w[1] / std[1])
axes[2].plot(xr, xb, 'y--', lw=2, label='Decision Boundary')
axes[2].set_title('Decision Boundary', fontsize=14, fontweight='bold')
axes[2].legend(fontsize=9)
axes[2].set_xlim(X[:, 0].min() - 1, X[:, 0].max() + 1)
axes[2].set_ylim(X[:, 1].min() - 0.01, X[:, 1].max() + 0.02)

plt.tight_layout()
plt.show()
print("\n🎉 그래프가 나오면 성공입니다!")
